# M01 — Correcciones de estaciones gravimétricas

**Level 2 · Prerrequisitos: notebook de fundamentos; lectura: 35 minutos.**

## Objetivos

Esta lección ejecuta el procesador de estaciones del producto sobre su control analítico versionado. Calcula perturbación de Bouguer, no inversión de densidad. Gravedad positiva hacia abajo en mGal; alturas elipsoidales WGS84 positivas hacia arriba. El control original no es dato de campo.

## Preparación

Use el entorno pipeline Python 3.12 ya revisado. Este notebook no instala nada.

Importe el código del repositorio y confirme su archivo fuente.

In [ ]:
import json, sys
from pathlib import Path
root=Path.cwd(); sys.path.insert(0,str(root/'data-pipeline'))
from gravity_processing import process_survey
# Identify the implementation loaded from this repository.
print('Módulo del producto:',Path(__import__('gravity_processing').__file__).resolve())

## Datos

Esta lección ejecuta el procesador de estaciones del producto sobre su control analítico versionado. Calcula perturbación de Bouguer, no inversión de densidad. Gravedad positiva hacia abajo en mGal; alturas elipsoidales WGS84 positivas hacia arriba. El control original no es dato de campo.

**Files:** docs/methods/gravity-processing/examples/station-control.json

## Método

Para latitud φ y altura del receptor h, D=g−γ(φ,h). La atracción de placa terrestre infinita es B=2πGρt×10⁵ mGal, con G en m³ kg⁻¹ s⁻², ρ en kg m⁻³ y espesor superficial t en m. La salida es D_B=D−B. Altura del receptor y espesor de terreno son distintos; no es corrección automática de aire libre ni DEM.

Ejecute el método del producto sobre el control declarado.

In [ ]:
request=json.loads((root/'docs/methods/gravity-processing/examples/station-control.json').read_text(encoding='utf-8'))
# Execute the product correction core on its committed analytic control.
result=process_survey(request['dataset'],request['config'])
print({'estado':result['dataset']['state'],'gravedad_derivada_mGal':result['qc']['derived_mgal'],'incertidumbre_mGal':result['processing']['uncertainty_mgal'],'aceptado':result['processing']['full_method_accepted']})

Control adversarial/negativo: aísle un supuesto o incumpla una restricción física declarada.

In [ ]:
try:
    process_survey(result['dataset'],request['config'])
except ValueError as error:
    # Reapplying an already reached correction state must be refused.
    print('Rechazo esperado por corrección duplicada:',str(error))
else:
    raise AssertionError('Repeated correction was accepted')

## Estado del arte

USGS describe adquisición y QA previas. Boule calcula gravedad normal dependiente de altura; Harmonica implementa atracción de placa. La incertidumbre condicional propaga errores marginales declarados; no es confianza geológica. Una estación analítica no evalúa mapas regionales, terreno ni elegibilidad de campo.

## Resultados

Se esperan cerca de 12 mGal, una historia ordenada y `full_method_accepted=false`. Reaplicar debe fallar. Esto es evidencia local de aritmética/estado, no aceptación M01 completa.

## Resumen

Una corrección es transformación tipada y ordenada. Conserve unidades, referencia, valor original e historia; nunca infiera metadatos faltantes por magnitud.

## Referencias

- https://www.fatiando.org/boule/v0.5.0/user_guide/normal_gravity.html
- https://doi.org/10.1190/1.1487109
- https://www.fatiando.org/harmonica/v0.7.0/api/generated/harmonica.bouguer_correction.html
- https://pubs.usgs.gov/tm/02/d04/tm2d4.pdf